# KuaiRand-1K：准备 → 第一阶段验收 → 第二阶段 B0/R0

第一次使用请逐格点击左侧 ▶，**不要点“全部运行”**。本 notebook 不会自动跑后续 A1–A5 正式实验。

1. 把运行时切换到 GPU（菜单 Runtime / 运行时 → Change runtime type / 更改运行时类型）。优先使用有足够 RAM 的 GPU runtime；A100 若可用可优先选，没有则先做检查。
2. 第 1–8 步是准备/验收。到 `READY FOR B0` 后，再决定何时启动长训练。
3. 第 9、10 步默认关闭；你手动把开关改成 True 才启动 B0、R0。
4. 任意步骤报错就停在该格，按最后“反馈包”步骤把结果发回，不用自行改 batch/lr。

正式配置：Adam lr=.001，batch=256，τ=.1，dropout=0，BN=false，Residual=0，No Correction，最多20轮、patience=3。B0=shallow+mean；R0=deep+Attention。

GPU 训练尚未在本运行包上执行；本地验证记录在 ZIP 的 verification 文件夹。小样本 PASS 只证明流程，不是正式性能。


## 第 1 步：上传完整运行包并核验源码

运行下格，选择你刚下载的 `KuaiRand_Stage12_Ready_Pack.zip`，**不要选原始旧 ablation ZIP**。
运行包里已有代码和配置，无需从 GitHub 手工复制。重连后重新上传同一份 ZIP 即可。


In [ ]:
from google.colab import files
from pathlib import Path
import json, hashlib, zipfile, sys, subprocess, os
uploaded = files.upload()
names=[name for name in uploaded if name.endswith('.zip')]
assert len(names)==1, '请只上传这一个 ZIP 运行包'
with zipfile.ZipFile(names[0]) as archive:
    for item in archive.infolist():
        destination=(Path('/content')/item.filename).resolve()
        assert destination.is_relative_to(Path('/content')), 'Unsafe archive path'
    archive.extractall('/content')
PROJECT=Path('/content/architecture_stage12/training')
freeze=json.loads((PROJECT/'source_checksums.json').read_text())
for relative, expected in freeze.items():
    assert hashlib.sha256((PROJECT/relative).read_bytes()).hexdigest()==expected, relative
print('SOURCE CHECK PASS:', len(freeze), 'files verified')


## 第 2 步：挂载 Drive 与授权数据读取

点击 Google 授权时，用有共享 SAR 文件夹权限的账号。输出保存在你自己的 `MyDrive/KuaiRank/architecture-v1` 下，原团队文件不会被覆盖。


In [ ]:
from google.colab import drive, auth
from googleapiclient.discovery import build
import google.auth
drive.mount('/content/drive')
auth.authenticate_user()
credentials,_=google.auth.default()
service=build('drive','v3',credentials=credentials,cache_discovery=False)
BASE=Path('/content/drive/MyDrive/KuaiRank/architecture-v1')
DATA_DIR=BASE/'data'
RUN_ROOT=BASE/'runs'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
print('DRIVE READY:', BASE)


## 第 3 步：安装冻结环境与确认 GPU

第一次下载环境可能较慢。使用独立 Python 3.11 环境，不要求你重启 notebook kernel。
成功后看到 `ENVIRONMENT PASS`、GPU 名称和显存。若 CUDA unavailable，回菜单选择 GPU 后重连。


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','uv','PyYAML==6.0.3'],check=True)
subprocess.run(['uv','sync','--locked','--directory',str(PROJECT),'--python','3.11'],check=True)
PYTHON=str(PROJECT/'.venv/bin/python')
sys.path.insert(0,str(PROJECT))
from architecture import stream
stream([PYTHON,'-u','-c',
    "import json,platform,torch,numpy,pyarrow; assert torch.cuda.is_available(), 'Connect a GPU runtime'; print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,'numpy':numpy.__version__,'pyarrow':pyarrow.__version__,'gpu':torch.cuda.get_device_name(),'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30}))"],RUN_ROOT/'environment.log')
last=(RUN_ROOT/'environment.log').read_text().strip().splitlines()[-1]
(RUN_ROOT/'environment.json').write_text(last+'\n')
print('ENVIRONMENT PASS')


## 第 4 步：设置 W&B Secret

打开 Colab 左侧 🔑 Secrets → Add new secret：名称填写 `WANDB_API_KEY`，值填写你在 W&B 账号设置中取得的 API key，开启“Notebook access”。**不要把 key 发给我，也不要直接写进代码。**

下面使用原 SAR notebook 已记录的 `https://api.wandb.ai`。若你实际账号只在其它服务器上，需先核对那个服务器的 API 地址；不要直接把组织网页地址当 API 地址。出现权限/登录错误时，把报错文字发回即可。


In [ ]:
from google.colab import userdata
os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY')
assert os.environ['WANDB_API_KEY'], 'Secret 不能为空'
os.environ['WANDB_BASE_URL']='https://api.wandb.ai'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('SECRET READY（不会显示密钥）')


## 第 5 步：第一阶段——小样本测试、断线恢复与上传验收

不下载大数据，先用合成小样本检查六种架构与 B0/R0 中断恢复，并向 W&B 上传一个标记为 SMOKE-ONLY 的 checkpoint。此处 Recall=1 也不代表真实效果。

成功后必须出现 `STAGE 1 PASS`，且 online_artifact_verified=true。
若失败，请保留目录；修复后需要重做 smoke 时，将 `SMOKE_NAME` 改为新的名字，例如 `smoke-retry1`。


In [ ]:
SMOKE_NAME='smoke'
SMOKE_DIR=RUN_ROOT/SMOKE_NAME
pass_file=SMOKE_DIR/'stage1_pass.json'
if pass_file.exists():
    print('Reusing existing smoke PASS:', pass_file)
else:
    stream([PYTHON,'-u','-m','unittest','-v','test_training','test_logq','test_stage12'],RUN_ROOT/'stage1_tests.log')
    stream([PYTHON,'-u','check_stage1.py','--device','cuda','--online','--output',str(SMOKE_DIR)],RUN_ROOT/'stage1_smoke.log')
smoke=json.loads(pass_file.read_text())
assert smoke['status']=='PASS' and smoke['online_artifact_verified']
assert smoke['device']=='cuda'
current_env=json.loads((RUN_ROOT/'environment.json').read_text())
current_sha=subprocess.check_output([PYTHON,'-c','from kt1k.engine import code_hash; print(code_hash())'],cwd=PROJECT,text=True).strip()
assert smoke['torch']==current_env['torch'] and smoke['gpu']==current_env['gpu'], 'GPU/环境改变，请更换SMOKE_NAME重新验收'
assert smoke['reports']['B0']['code_sha256']==current_sha, '源码改变，请更换SMOKE_NAME重新验收'
print('STAGE 1 PASS: GPU + exact resume + online Artifact')


## 第 6 步：准备真实 1K train/validation 数据

会从团队共享 Drive 读取已冻结的数据并校验 SHA256，保存到你自己的目录。数据已存在且一致时跳过下载。不会下载 sealed test 目标。

成功输出 `DATA READY`。下载 403/404 常见原因是当前账号没有团队文件权限；将错误发回，先不要换成其他数据集。


In [ ]:
from fetch_data import fetch
report=fetch(service,DATA_DIR,PROJECT/'data_release.json')
assert report['test_targets_downloaded'] is False
(RUN_ROOT/'data_audit.json').write_text(json.dumps(report,indent=2)+'\n')
print('DATA READY:',report)


## 第 7 步：真实数据零更新 GPU 检查（六个配置）

每组会显示较长的特征检查信息。不是正式训练，不更新权重。
必须看到 user_mlp_input=131、item_mlp_input=64，train_queries=1335793、validation_queries=555677，No Correction。


In [ ]:
ALL_IDS=['B0','R0','A1','A2','A3','A5']
stream([PYTHON,'-u','architecture.py','preflight','--data',str(DATA_DIR),'--output-root',str(RUN_ROOT),'--ids',*ALL_IDS],RUN_ROOT/'real_preflight.log')
for name in ALL_IDS:
    record=json.loads((RUN_ROOT/name/'seed-42'/'preflight'/'feature_acceptance.json').read_text())
    assert record['passed'] and record['gpu_feature_acceptance']
    assert (record['user_mlp_input'],record['item_mlp_input'])==(131,64)
    assert record['train_queries']==1335793 and record['validation_queries']==555677
    assert record['sampling_correction']['mode']=='none'
    print(name,record['history_encoder'],record['cuda_device'],record['validation_target_coverage'])
print('REAL PREFLIGHT PASS')


## 第 8 步：真实数据短程探测（每配置200次更新）

在独立 `probe` 目录运行，**不把短程权重带入正式训练**。这一小段通常还没有跑完一个 epoch，因此暂时没有 Recall/NDCG 是正常的。
六组都通过后出现 `READY FOR B0`。先运行后面的反馈格，把反馈包发给我核对也可以；不必立即花时间跑完整训练。


In [ ]:
stream([PYTHON,'-u','architecture.py','probe','--data',str(DATA_DIR),'--output-root',str(RUN_ROOT),'--ids',*ALL_IDS,'--steps','200'],RUN_ROOT/'real_probe.log')
for name in ALL_IDS:
    m=json.loads((RUN_ROOT/name/'seed-42'/'probe'/'run_manifest.json').read_text())
    assert m['global_step']>=200
sizes={name:(RUN_ROOT/name/'seed-42'/'probe'/'latest.pt').stat().st_size for name in ['B0','R0']}
needed=int(1.3*22*sum(sizes.values())+1024**3)
quota=service.about().get(fields='storageQuota').execute().get('storageQuota',{})
free=int(quota['limit'])-int(quota.get('usage',0)) if quota.get('limit') else None
budget={'estimated_additional_drive_bytes_B0_R0':needed,'drive_free_bytes':free,'probe_checkpoint_bytes':sizes}
(RUN_ROOT/'resource_budget.json').write_text(json.dumps(budget,indent=2)+'\n')
print('B0+R0 checkpoint 空间预估(含余量) GiB:',round(needed/1024**3,2))
if free is not None:
    print('Drive 当前剩余 GiB:',round(free/1024**3,2))
    assert free>=needed, '空间可能不足。请保留结果并发反馈，不要先启动正式训练。'
print('READY FOR B0: 六组真实数据 preflight/probe 已通过')


## 第 9 步：第二阶段——正式 B0 seed42

只有第1–8步全部成功后，把 `START_B0=False` 改成 `START_B0=True` 再运行这格。

最多20轮，连续3轮验证指标不改善就提前停止。`phase=validation` 后训练 step 暂停增长很正常；`catalog encoding` 和 `evaluation x/555677` 表示正在做全目录验证。

每 epoch 会输出 loss、Recall/NDCG、epoch checkpoint 和 Artifact 上传结果。断线后重新执行1–4步、已有检查按需复核，再运行这格，自动从 latest.pt 恢复。


In [ ]:
START_B0=False
if START_B0:
    budget=json.loads((RUN_ROOT/'resource_budget.json').read_text())
    assert budget['drive_free_bytes'] is None or budget['drive_free_bytes']>=budget['estimated_additional_drive_bytes_B0_R0'], '先解决Drive空间不足'
    assert json.loads((SMOKE_DIR/'stage1_pass.json').read_text())['online_artifact_verified']
    for name in ALL_IDS:
        assert json.loads((RUN_ROOT/name/'seed-42'/'probe'/'run_manifest.json').read_text())['global_step']>=200
    stream([PYTHON,'-u','architecture.py','train','--data',str(DATA_DIR),'--output-root',str(RUN_ROOT),'--ids','B0'],RUN_ROOT/'B0_console.log')
    print('B0 FINISHED; 请运行反馈包格')
else:
    print('B0 尚未启动。准备好后把 START_B0 改为 True。')


## 第 10 步：正式 R0 seed42

B0 正常完成后，再把 `START_R0` 改成 True。R0 从头训练，不加载 B0 权重。
R0 完成后给 Annabel 的是同版本代码、r0.yaml、结果与 checkpoint 索引，不是让她重新跑一份 R0。


In [ ]:
START_R0=False
if START_R0:
    b0=json.loads((RUN_ROOT/'B0'/'seed-42'/'run'/'run_manifest.json').read_text())
    assert b0['complete'], '先完成 B0'
    stream([PYTHON,'-u','architecture.py','train','--data',str(DATA_DIR),'--output-root',str(RUN_ROOT),'--ids','R0'],RUN_ROOT/'R0_console.log')
    print('R0 FINISHED; 请运行反馈包格')
else:
    print('R0 尚未启动。B0完成后把 START_R0 改为 True。')


## 第 11 步：下载你需要发给我的反馈包

阶段1/短程探测完成后发一次；B0完成后发一次；R0完成后再发一次。出错时也可运行这格。
反馈包只含配置、检查、聚合指标与日志，不含原始数据、API key 或模型权重。直接把下载的 `SEND_TO_CHATGPT.zip` 上传到当前聊天，不用手抄数字。

若错误发生在 PROJECT/RUN_ROOT 定义之前，先发该格完整红色报错文字即可。运行时重启后，先重新执行前面的初始化格。


In [ ]:
from architecture import feedback_bundle
feedback=feedback_bundle(RUN_ROOT)
from google.colab import files
files.download(str(feedback))


## 故障恢复：只在出现 Task occupied 且你确认旧进程已停止时使用

同一个 config/seed 不能在两个 runtime 同时跑。训练格正在运行时，不能运行下面的解锁格。
断开 notebook 网页不一定代表后端进程停止，请先在 Colab 的“管理会话”里确认旧 session 已终止。不要删除 checkpoint 或更改配置来绕过绑定。


In [ ]:
CONFIRM_OLD_RUNTIME_STOPPED=False
TASK_TO_UNLOCK='B0'  # 若占用的是R0，改成R0
if CONFIRM_OLD_RUNTIME_STOPPED:
    lock=RUN_ROOT/TASK_TO_UNLOCK/'seed-42'/'task.lock'
    if lock.exists():
        print((lock/'owner.json').read_text())
        (lock/'owner.json').unlink(missing_ok=True)
        lock.rmdir()
        print('LOCK REMOVED; 重新执行对应训练格即可恢复')
else:
    print('未解锁；只有确认旧runtime已停止时才把开关改为True')
